In [4]:
import requests
import pandas as pd
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

try:
    teams_df = pd.read_sql(
        "SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev",
        conn,
    ).dropna(subset=["team_abbrev"])
finally:
    conn.close()

player_records = []

for team in teams_df.itertuples(index=False):
    url = f"https://api-web.nhle.com/v1/roster/{team.team_abbrev}/current"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        roster = response.json()
    except requests.RequestException as error:
        print(f"Could not load {team.team_abbrev}: {error}")
        continue

    for roster_position in ("forwards", "defensemen", "goalies"):
        for player in roster.get(roster_position, []):
            first_name = player.get("firstName", {})
            last_name = player.get("lastName", {})
            player_records.append({
                "player_id": player.get("id"),
                "team_id": team.team_id,
                "first_name": first_name.get("default", "") if isinstance(first_name, dict) else first_name,
                "last_name": last_name.get("default", "") if isinstance(last_name, dict) else last_name,
                "position": player.get("positionCode", roster_position),
                "jersey_number": player.get("sweaterNumber"),
                "birth_date": player.get("birthDate"),
                "birth_country": player.get("birthCountry"),
                "height_cm": player.get("heightInCentimeters"),
                "weight_kg": player.get("weightInKilograms"),
                "shoots_catches": player.get("shootsCatches"),
                "headshot_url": player.get("headshot"),
            })

players_df = pd.DataFrame(player_records).drop_duplicates("player_id")
players_df = players_df[
    [
        "player_id",
        "team_id",
        "first_name",
        "last_name",
        "position",
        "jersey_number",
        "birth_date",
        "birth_country",
        "height_cm",
        "weight_kg",
        "shoots_catches",
        "headshot_url",
    ]
]
players_df

C:\Users\nihaa\AppData\Local\Temp\ipykernel_57964\3233857491.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  teams_df = pd.read_sql(


,player_id,team_id,first_name,last_name,position,jersey_number,birth_date,birth_country,height_cm,weight_kg,shoots_catches,headshot_url
0,8484153,18,Leo,Carlsson,C,91.0,2004-12-26,SWE,191,94,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
1,8481538,18,Judd,Caulfield,R,28.0,2001-03-19,USA,191,100,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
2,8482118,18,Sam,Colangelo,R,12.0,2001-12-26,USA,188,97,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
3,8482081,18,Sean,Farrell,C,59.0,2001-11-02,USA,175,83,L,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
4,8483444,18,Nathan,Gaucher,C,41.0,2003-11-06,CAN,191,103,R,https://assets.nhle.com/mugs/nhl/20262027/ANA/...
...,...,...,...,...,...,...,...,...,...,...,...,...
1263,8478911,12,Matt,Roy,D,3.0,1995-03-01,USA,188,100,R,https://assets.nhle.com/mugs/nhl/20262027/WSH/...
1264,8480873,12,Rasmus,Sandin,D,38.0,2000-03-07,SWE,180,86,L,https://assets.nhle.com/mugs/nhl/20262027/WSH/...
1265,8479292,12,Charlie,Lindgren,G,79.0,1993-12-18,USA,188,86,R,https://assets.nhle.com/mugs/nhl/20262027/WSH/...
1266,8483532,12,Clay,Stevenson,G,33.0,1999-03-03,CAN,193,88,L,https://assets.nhle.com/mugs/nhl/20262027/WSH/...


In [5]:
conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

try:
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS player (
            player_id BIGINT PRIMARY KEY,
            team_id INT NOT NULL,
            first_name VARCHAR(100),
            last_name VARCHAR(100),
            position VARCHAR(10),
            jersey_number INT,
            birth_date DATE,
            birth_country VARCHAR(3),
            height_cm DECIMAL(5, 2),
            weight_kg DECIMAL(5, 2),
            shoots_catches VARCHAR(1),
            headshot_url TEXT
        )
    """)

    player_columns = [
        "player_id",
        "team_id",
        "first_name",
        "last_name",
        "position",
        "jersey_number",
        "birth_date",
        "birth_country",
        "height_cm",
        "weight_kg",
        "shoots_catches",
        "headshot_url",
    ]
    player_records = list(
        players_df[player_columns]
        .astype(object)
        .where(pd.notna(players_df[player_columns]), None)
        .itertuples(index=False, name=None)
    )

    insert_query = """
        INSERT INTO player (
            player_id, team_id, first_name, last_name, position,
            jersey_number, birth_date, birth_country, height_cm,
            weight_kg, shoots_catches, headshot_url
        )
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
        ON DUPLICATE KEY UPDATE
            team_id = VALUES(team_id),
            first_name = VALUES(first_name),
            last_name = VALUES(last_name),
            position = VALUES(position),
            jersey_number = VALUES(jersey_number),
            birth_date = VALUES(birth_date),
            birth_country = VALUES(birth_country),
            height_cm = VALUES(height_cm),
            weight_kg = VALUES(weight_kg),
            shoots_catches = VALUES(shoots_catches),
            headshot_url = VALUES(headshot_url)
    """

    cursor.executemany(insert_query, player_records)
    conn.commit()
    print(f"Loaded {len(player_records)} players into the player table")
finally:
    conn.close()

Loaded 1268 players into the player table
